In [1]:
from scripts.wgs_stats import Gene
import numpy as np
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

/home/crehmann/mambaforge/envs/biopython/lib/python3.9/site-packages/Bio/codonalign/__init__.py:21: BiopythonExperimentalWarning: Bio.codonalign is an experimental module which may undergo significant changes prior to its future official release.
  warnings.warn(


In [17]:
gene1 = 'AGAP029767'
species1 = 'gamb'
vcf1 = '/sietch_colab/data_share/Ag1000G/Ag3.0/vcf/unphased_vcf/gamb/ag1000g.agam_n1470.merged_allsites.vcf.gz'
filter1 = '/sietch_colab/data_share/Ag1000G/Ag3.0/vcf/unphased_vcf/gamb/ag1000g.agam_n1470.merged_variants.sitefilt.vcf.gz'
fasta1 = 'data/gamb/VectorBase-68_AgambiaePEST_Genome.fasta'
annotation1 = 'data/gamb/VectorBase-68_AgambiaePEST.db'

gene2 = 'AFUN2_003877'
species2 = 'afun'
vcf2 = '/sietch_colab/crehmann/vo_afun_release/v1.0/merged_vcf/afun_merged.vcf.gz'
filter2 = '/sietch_colab/crehmann/vo_afun_release/v1.0/site_filters/dt_20200416/vcf/funestus/merged_funestus_sitefilters.vcf.gz'
fasta2 = 'data/afun/VectorBase-68_AfunestusAfunGA1_Genome.fasta'
annotation2 = 'data/afun/VectorBase-68_AfunestusAfunGA1.db'

In [18]:
gene1 = Gene(name = gene1)
gene1.fetch_gene_coordinates(annotation1, fasta1)
gene1.fetch_variation(vcf_file = vcf1,
                      annotation = annotation1,
                      fasta = fasta1,
                      filter = filter1)

Fetching transcripts for gene AGAP029767 on chromosome 2R
Fetching coding sequence for transcript AGAP029767-RA


In [19]:
gene2 = Gene(name = gene2)
gene2.chromosome = f'AfunGA1_{gene2.chromosome}'
gene2.fetch_gene_coordinates(annotation2, fasta2)
gene2.fetch_variation(vcf_file = vcf2,
                      annotation = annotation2,
                      fasta = fasta2,
                      filter = filter2)

Fetching transcripts for gene AFUN2_003877 on chromosome 2RL
Fetching coding sequence for transcript AFUN2_003877.R5497
Fetching coding sequence for transcript AFUN2_003877.R5498
Fetching coding sequence for transcript AFUN2_003877.R5499


In [20]:
# generate seqio records of nucleotides and amino acids
species_list = []
nt_records = []
aa_records = []

for gene, species in zip([gene1, gene2], [species1, species2]):
    for i in range(gene.transcripts[list(gene.transcripts.keys())[0]]['sequences']['nt_seq_array'].shape[1]):
        species_list.append(species)
        species_list.append(species)
        nt_seq = gene.transcripts[list(gene.transcripts.keys())[0]]['sequences']['nt_seq_array'][:,i]
        nt_records.append(SeqRecord(Seq(''.join(nt_seq[:,0])), id=f"{gene.name}_{species}_nt_{i}.0", description=""))
        nt_records.append(SeqRecord(Seq(''.join(nt_seq[:,1])), id=f"{gene.name}_{species}_nt_{i}.1", description=""))

    for i in range(gene.transcripts[list(gene.transcripts.keys())[0]]['sequences']['aa_seq_array'].shape[1]):
        aa_seq = gene.transcripts[list(gene.transcripts.keys())[0]]['sequences']['aa_seq_array'][:,i]
        aa_records.append(SeqRecord(Seq(''.join(aa_seq[:,0])), id=f"{gene.name}_{species}_aa_{i}.0", description=""))
        aa_records.append(SeqRecord(Seq(''.join(aa_seq[:,1])), id=f"{gene.name}_{species}_aa_{i}.1", description=""))

In [21]:
import subprocess
from Bio import SeqIO
# save protein fasta and align it
SeqIO.write(aa_records, 'test_aa.fasta', 'fasta')
cmd = "muscle -super5 test_aa.fasta -output test_aa_aln.txt"
result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, text=True)


muscle 5.3.linux64 []  1056Gb RAM, 256 cores
Built Jul 30 2025 21:13:04
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

[super5 test_aa.fasta]
Input: 4252 seqs, avg length 722, max 726, min 712

00:00 15Mb    100.0% Derep 1021 uniques, 3231 dupes
00:00 15Mb   CPU has 256 cores, running 256 threads                  
01:29 19Mb    100.0% UCLUST 1021 seqs EE<0.01, 2 centroids, 1018 members
01:29 19Mb    100.0% UCLUST 2 seqs EE<0.30, 1 centroids, 0 members      
01:29 2.2Gb   100.0% Make cluster MFAs                            
1 clusters pass 1                     
1 clusters pass 2
01:29 2.2Gb  
01:29 2.2Gb  Align cluster 1 / 1 (2 seqs)
01:29 2.2Gb  
01:29 2.2Gb   100.0% Derep 2 uniques, 0 dupes
01:29 2.2Gb   100.0% Calc posteriors         
01:29 2.2Gb   100.0% UPGMA5         
01:29 2.2Gb   100.0% Consensus sequences
Inserting 3231 dupes... done.           


In [22]:
# save nt records as fasta and read back in just for ease of working with it
SeqIO.write(nt_records, 'test_nt.fasta', 'fasta')
nt_recs = SeqIO.index('test_nt.fasta', 'fasta')

In [23]:
from Bio import AlignIO
from Bio import Align
from Bio.Align import CodonAligner
aligner = CodonAligner()

protein_aln = Align.read('test_aa_aln.txt', 'fasta')
codon_alignments = []
for prec in protein_aln.sequences:
    ntrec = nt_recs[prec.id.replace('_aa_', '_nt_')]
    alignments = aligner.align(prec, ntrec)
    assert len(alignments) == 1
    codon_alignment = next(alignments)
    codon_alignments.append(codon_alignment)

In [24]:
alignment = protein_aln.mapall(codon_alignments)

In [25]:
from Bio.Data import CodonTable
from collections import defaultdict
from heapq import heapify
from heapq import heappop
from heapq import heappush
from math import floor

def _get_codon2codon_matrix(codon_table):
    ## https://github.com/biopython/biopython/blob/master/Bio/Align/analysis.py
 
    """Get codon codon substitution matrix (PRIVATE).

    Elements in the matrix are number of synonymous and nonsynonymous
    substitutions required for the substitution.
    """
    bases = ("A", "T", "C", "G")
    codons = [
        codon
        for codon in list(codon_table.forward_table.keys()) + codon_table.stop_codons
        if "U" not in codon
    ]
    # set up codon_dict considering stop codons
    codon_dict = codon_table.forward_table.copy()
    for stop in codon_table.stop_codons:
        codon_dict[stop] = "stop"
    # count site
    num = len(codons)
    G = {}  # graph for substitution
    nonsyn_G = {}  # graph for nonsynonymous substitution
    graph = {}
    graph_nonsyn = {}
    for i, codon in enumerate(codons):
        graph[codon] = {}
        graph_nonsyn[codon] = {}
        for p in range(3):
            for base in bases:
                tmp_codon = codon[0:p] + base + codon[p + 1 :]
                if codon_dict[codon] != codon_dict[tmp_codon]:
                    graph_nonsyn[codon][tmp_codon] = 1
                    graph[codon][tmp_codon] = 1
                else:
                    if codon != tmp_codon:
                        graph_nonsyn[codon][tmp_codon] = 0.1
                        graph[codon][tmp_codon] = 1
    for codon1 in codons:
        nonsyn_G[codon1] = {}
        G[codon1] = {}
        for codon2 in codons:
            if codon1 == codon2:
                nonsyn_G[codon1][codon2] = 0
                G[codon1][codon2] = 0
            else:
                nonsyn_G[codon1][codon2] = _dijkstra(graph_nonsyn, codon1, codon2)
                G[codon1][codon2] = _dijkstra(graph, codon1, codon2)
    return G, nonsyn_G

def _dijkstra(graph, start, end):
    """Dijkstra's algorithm Python implementation (PRIVATE).

    Algorithm adapted from
    http://thomas.pelletier.im/2010/02/dijkstras-algorithm-python-implementation/.
    However, an obvious bug in::

        if D[child_node] >(<) D[node] + child_value:

    is fixed.
    This function will return the distance between start and end.

    Arguments:
     - graph: Dictionary of dictionary (keys are vertices).
     - start: Start vertex.
     - end: End vertex.

    Output:
       List of vertices from the beginning to the end.

    """
    D = {}  # Final distances dict
    P = {}  # Predecessor dict
    # Fill the dicts with default values
    for node in graph.keys():
        D[node] = 100  # Vertices are unreachable
        P[node] = ""  # Vertices have no predecessors
    D[start] = 0  # The start vertex needs no move
    unseen_nodes = list(graph.keys())  # All nodes are unseen
    while len(unseen_nodes) > 0:
        # Select the node with the lowest value in D (final distance)
        shortest = None
        node = ""
        for temp_node in unseen_nodes:
            if shortest is None:
                shortest = D[temp_node]
                node = temp_node
            elif D[temp_node] < shortest:
                shortest = D[temp_node]
                node = temp_node
        # Remove the selected node from unseen_nodes
        unseen_nodes.remove(node)
        # For each child (ie: connected vertex) of the current node
        for child_node, child_value in graph[node].items():
            if D[child_node] > D[node] + child_value:
                D[child_node] = D[node] + child_value
                # To go to child_node, you have to go through node
                P[child_node] = node
        if node == end:
            break
    # Set a clean path
    path = []
    # We begin from the end
    node = end
    distance = 0
    # While we are not arrived at the beginning
    while not (node == start):
        if path.count(node) == 0:
            path.insert(0, node)  # Insert the predecessor of the current node
            node = P[node]  # The current node becomes its predecessor
        else:
            break
    path.insert(0, start)  # Finally, insert the start vertex
    for i in range(len(path) - 1):
        distance += graph[path[i]][path[i + 1]]
    return distance

def _count_replacement(codons, G):
    """Count replacement needed for a given codon_set (PRIVATE)."""
    if len(codons) == 1:
        return 0, 0
    elif len(codons) == 2:
        codons = list(codons)
        return floor(G[codons[0]][codons[1]])
    else:
        subgraph = {
            codon1: {codon2: G[codon1][codon2] for codon2 in codons if codon1 != codon2}
            for codon1 in codons
        }
        return _prim(subgraph)


def _prim(G):
    """Prim's algorithm to find minimum spanning tree (PRIVATE).

    Code is adapted from
    http://programmingpraxis.com/2010/04/09/minimum-spanning-tree-prims-algorithm/
    """
    nodes = []
    edges = []
    for i in G.keys():
        nodes.append(i)
        for j in G[i]:
            if (i, j, G[i][j]) not in edges and (j, i, G[i][j]) not in edges:
                edges.append((i, j, G[i][j]))
    conn = defaultdict(list)
    for n1, n2, c in edges:
        conn[n1].append((c, n1, n2))
        conn[n2].append((c, n2, n1))
    mst = []  # minimum spanning tree
    used = set(nodes[0])
    usable_edges = conn[nodes[0]][:]
    heapify(usable_edges)
    while usable_edges:
        cost, n1, n2 = heappop(usable_edges)
        if n2 not in used:
            used.add(n2)
            mst.append((n1, n2, cost))
            for e in conn[n2]:
                if e[2] not in used:
                    heappush(usable_edges, e)
    length = 0
    for p in mst:
        length += floor(p[2])
    return length


In [26]:
codon_table = CodonTable.generic_by_id[1]
G, nonsyn_G = _get_codon2codon_matrix(codon_table=codon_table)

In [27]:
unique_species = list(set(species_list))
sequences = []
for sequence in alignment.sequences:
    sequences.append(str(sequence.seq))

In [28]:
import sys
syn_fix, nonsyn_fix, syn_poly, nonsyn_poly = 0, 0, 0, 0
starts = sys.maxsize
for ends in alignment.coordinates.transpose():
    step = min(ends - starts)
    for j in range(0, step, 3):
        codons = {key: [] for key in unique_species}
        for key, sequence, start in zip(species_list, sequences, starts):
            codon = sequence[start + j : start + j + 3]
            codons[key].append(codon)
        fixed = True
        all_codons = set()
        for value in codons.values():
            value = set(value)
            if len(value) > 1:
                fixed = False
            all_codons.update(value)
        if len(all_codons) == 1:
            continue
        nonsyn = _count_replacement(all_codons, nonsyn_G)
        syn = _count_replacement(all_codons, G) - nonsyn
        if fixed is True:
            # fixed
            nonsyn_fix += nonsyn
            syn_fix += syn
        else:
            # not fixed
            nonsyn_poly += nonsyn
            syn_poly += syn    
    starts = ends


In [29]:
syn_fix, nonsyn_fix, syn_poly, nonsyn_poly

(36, 4, 1066, 696)

In [31]:
alpha = 1 - ((syn_fix * nonsyn_poly) / (nonsyn_fix * syn_poly))
alpha

-4.876172607879925

In [32]:
from scipy.stats import chi2_contingency
tab = np.array([[syn_fix, syn_poly], [nonsyn_fix, nonsyn_poly]])
chi2, p, dof, ex = chi2_contingency(tab)
p

0.0002931571612117006